# EXP-006 — Thiên lệch chủng tộc trên RAF-DB
Tách RAF-DB test theo nhãn chủng tộc do người gán (`basic/Annotation/manual.zip`: 0 Caucasian · 1 African-American · 2 Asian), tính UAR/WAR zero-shot riêng từng nhóm.
- So sánh: **CLIP gốc** (không head) · **Paper** (checkpoint trong repo) · **EXP-000** (3 seed) · **EXP-004** λ = 1 ep30 (3 seed).
- Không train. Feature CLIP của ảnh/text tính **1 lần**, mỗi head chỉ là một phép nhân ma trận → chạy nhanh.
- Tiền xử lý giống `test.py`: `Resize((224, 224))` + `ToTensor()`, prompt trung bình theo lớp.

Nhóm Asian chỉ có 483 ảnh (Anger 9, Fear 12) → thêm **UAR5** (bỏ Anger, Fear) và bootstrap CI cho Δ = Asian − Caucasian.

Tiêu chí: **có thiên lệch** nếu UAR(Asian) thấp hơn UAR(Caucasian) ≥ 3 điểm, ổn định qua 3 seed, và CI 95% không chứa 0.

# Cấu hình

In [ ]:
SEEDS = [1, 2, 3]
BRANCH = "exp/006-rafdb-race-bias"   # chỉ cần models/clip + models/Text.py (không cần lavis)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp006"
ATTR_CSV = f"{OUT_DIR}/rafdb_test_attributes.csv"   # trích từ basic/Annotation/manual.zip; không commit (giấy phép RAF-DB), tự upload
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"  # tìm đệ quy expclip_l14_face_seed{N}-model.pth
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
N_BOOT = 2000
MAIN_PROMPT = "Expression Ensemble Five"

PROMPT_NAMES = ["Class Name", "An Expression of Name", "A Photo of A Face with An Expression of Name",
                "Expression Ensemble Five", "Expression Ensemble Ten"]   # cùng thứ tự với FER_prompt_list
RACE = {0: "Caucasian", 1: "AfrAm", 2: "Asian"}
GENDER = {0: "Nam", 1: "Nữ", 2: "Không rõ"}
AGE = {0: "0-3", 1: "4-19", 2: "20-39", 3: "40-69", 4: "70+"}

# Mount Drive & GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(ATTR_CSV), f"chưa upload {ATTR_CSV}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Clone repo & giải nén test set

In [ ]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
!cd /content/Exp-CLIP && git checkout $BRANCH && git pull && git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/static_test_sets/RAFDB/test
# CLIP vendored cần ftfy và pkg_resources (setuptools mới đã bỏ)
!pip install -q ftfy "setuptools<81"

# Tìm checkpoint
Thiếu file nào thì bỏ qua (in cảnh báo).

In [ ]:
import glob

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
        else:
            print(f"{m} seed {s}: {p}")

# Feature CLIP L/14 (ảnh + text), tính 1 lần

In [ ]:
import sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets, torchvision.transforms as T
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list

clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
ds = datasets.ImageFolder("/content/static_test_sets/RAFDB/test",
                          T.Compose([T.Resize((224, 224)), T.ToTensor()]))   # giống test.py
loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
with torch.no_grad():
    img = torch.cat([clip_model.encode_image(x.cuda()).float().cpu() for x, _ in loader])
    txt = {pn: clip_model.encode_text(clip.tokenize(pl["RAFDB"], context_length=77, truncate=True).cuda()).float().cpu()
           for pn, pl in zip(PROMPT_NAMES, FER_prompt_list)}
y = np.array(ds.targets)
classes = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
print(img.shape, classes, {pn: tuple(t.shape) for pn, t in txt.items()})

# Ghép nhãn chủng tộc / giới tính / tuổi

In [ ]:
attr = pd.read_csv(ATTR_CSV).set_index("image")
names = [os.path.basename(p) for p, _ in ds.samples]
assert len(names) == len(attr) == 3068 and set(names) == set(attr.index), "tên ảnh không khớp với CSV"
attr = attr.loc[names]
assert (attr["emotion"].values == np.array(classes)[y]).all(), "nhãn cảm xúc không khớp"
race = attr["race"].map(RACE).values
gender = attr["gender"].map(GENDER).values
age = attr["age"].map(AGE).values
print(pd.crosstab(race, np.array(classes)[y], margins=True))

# Dự đoán & metric

In [ ]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, pn):
    i, t = (img, txt[pn]) if W is None else (img @ W.T, txt[pn] @ W.T)
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), 7, -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def per_class(y_, p_):
    return np.array([(p_[y_ == c] == c).mean() * 100 if (y_ == c).any() else np.nan for c in range(7)])

KEEP5 = [c for c in range(7) if classes[c] not in ("Anger", "Fear")]
def metrics(y_, p_):
    acc = per_class(y_, p_)
    return {"UAR": np.nanmean(acc), "UAR5": np.nanmean(acc[KEEP5]), "WAR": (p_ == y_).mean() * 100,
            **{f"acc_{classes[c]}": acc[c] for c in range(7)}}

GROUPS = {"Tất cả": np.ones(len(y), bool), **{r: race == r for r in RACE.values()}}
preds, rows = {}, []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for pn in PROMPT_NAMES:
            p = preds[(m, s, pn)] = predict(W, pn)
            for g, mask in GROUPS.items():
                rows.append({"model": m, "seed": s, "prompt": pn, "group": g, "n": int(mask.sum()), **metrics(y[mask], p[mask])})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp006_race_bias-results.csv", index=False)
np.savez_compressed(f"{OUT_DIR}/exp006_race_bias-preds.npz", names=np.array(names), y=y, race=race,
                    **{f"{m}|{s}|{pn}": p for (m, s, pn), p in preds.items()})
print(len(res), "dòng →", OUT_DIR)

# Kiểm tra: số 'Tất cả' phải khớp `test.py`
So với kết quả đã ghi của EXP-000 / EXP-004 (RAFDB). Lệch ≤ ~0.1 là do GPU fp16; lệch lớn thì pipeline sai, dừng lại.

In [ ]:
REF = {"EXP-000|1|Class Name": [50.36, 56.58], "EXP-000|1|An Expression of Name": [56.73, 64.05], "EXP-000|1|A Photo of A Face with An Expression of Name": [54.94, 65.12], "EXP-000|1|Expression Ensemble Five": [58.88, 66.88], "EXP-000|1|Expression Ensemble Ten": [57.73, 65.91], "EXP-000|2|Class Name": [49.61, 55.9], "EXP-000|2|An Expression of Name": [54.35, 62.26], "EXP-000|2|A Photo of A Face with An Expression of Name": [55.25, 64.44], "EXP-000|2|Expression Ensemble Five": [57.82, 65.78], "EXP-000|2|Expression Ensemble Ten": [56.65, 64.18], "EXP-000|3|Class Name": [50.73, 51.21], "EXP-000|3|An Expression of Name": [54.94, 59.68], "EXP-000|3|A Photo of A Face with An Expression of Name": [56.74, 64.77], "EXP-000|3|Expression Ensemble Five": [57.5, 63.3], "EXP-000|3|Expression Ensemble Ten": [57.19, 63.72], "EXP-004|1|Class Name": [54.65, 57.82], "EXP-004|1|An Expression of Name": [57.84, 62.68], "EXP-004|1|A Photo of A Face with An Expression of Name": [58.55, 68.68], "EXP-004|1|Expression Ensemble Five": [58.73, 65.94], "EXP-004|1|Expression Ensemble Ten": [58.86, 66.85], "EXP-004|2|Class Name": [53.74, 57.89], "EXP-004|2|An Expression of Name": [56.11, 61.8], "EXP-004|2|A Photo of A Face with An Expression of Name": [57.96, 67.89], "EXP-004|2|Expression Ensemble Five": [57.07, 64.77], "EXP-004|2|Expression Ensemble Ten": [57.95, 65.84], "EXP-004|3|Class Name": [52.44, 54.86], "EXP-004|3|An Expression of Name": [56.63, 61.86], "EXP-004|3|A Photo of A Face with An Expression of Name": [59.81, 68.29], "EXP-004|3|Expression Ensemble Five": [58.26, 65.06], "EXP-004|3|Expression Ensemble Ten": [58.5, 66.0]}

chk = []
for k, (u, w) in REF.items():
    m, s, pn = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.prompt == pn) & (res.group == "Tất cả")]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u, "dWAR": r.WAR.item() - w})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(5).to_string(index=False))

# Bảng 1 — Ensemble Five: UAR / UAR5 / WAR theo nhóm (mean ± std qua seed)

In [ ]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

main = res[res.prompt == MAIN_PROMPT]
for metric in ["UAR", "UAR5", "WAR"]:
    t = main.pivot_table(index="model", columns="group", values=metric, aggfunc=fmt)[["Tất cả", *RACE.values()]]
    d = main.pivot_table(index=["model", "seed"], columns="group", values=metric)
    t["Δ Asian−Cauc"] = (d["Asian"] - d["Caucasian"]).groupby("model").apply(fmt)
    print(f"--- {metric}\n{t.loc[list(HEADS)].to_string()}\n")

# Bảng 2 — Δ UAR (Asian − Caucasian) theo 5 kiểu prompt
Thiên lệch có ổn định khi đổi prompt không?

In [ ]:
d = res.pivot_table(index=["model", "seed", "prompt"], columns="group", values="UAR")
delta = (d["Asian"] - d["Caucasian"]).groupby(["model", "prompt"]).mean().unstack()[PROMPT_NAMES]
print(delta.loc[list(HEADS)].round(2).to_string())

# Bảng 3 — Accuracy từng lớp: Asian vs Caucasian (Ensemble Five, mean seed)
Lớp nào gây ra chênh lệch? (n nhỏ ở Anger/Fear → đọc thận trọng)

In [ ]:
cols = [f"acc_{c}" for c in classes]
n_cls = pd.crosstab(race, np.array(classes)[y])[classes]
print("Số ảnh:\n" + n_cls.loc[list(RACE.values())].to_string() + "\n")
for m in HEADS:
    t = main[(main.model == m) & main.group.isin(["Caucasian", "Asian"])].groupby("group")[cols].mean()
    t.columns = classes
    t.loc["Δ"] = t.loc["Asian"] - t.loc["Caucasian"]
    print(f"--- {m}\n{t.round(1).to_string()}\n")

# Bootstrap CI 95% cho Δ = Asian − Caucasian (Ensemble Five)
Lấy mẫu lại ảnh trong từng nhóm (cùng chỉ số cho mọi seed), Δ = trung bình qua seed.

In [ ]:
rng = np.random.default_rng(0)
ia, ic = np.where(race == "Asian")[0], np.where(race == "Caucasian")[0]
boot = []
for m, d in HEADS.items():
    P = [preds[(m, s, MAIN_PROMPT)] for s in d]
    draws = {"UAR": [], "UAR5": [], "WAR": []}
    for _ in range(N_BOOT):
        a, c = rng.choice(ia, len(ia)), rng.choice(ic, len(ic))
        ms = [(metrics(y[a], p[a]), metrics(y[c], p[c])) for p in P]
        for k in draws:
            draws[k].append(np.mean([ma[k] - mc[k] for ma, mc in ms]))
    for k, v in draws.items():
        point = np.mean([metrics(y[ia], p[ia])[k] - metrics(y[ic], p[ic])[k] for p in P])
        lo, hi = np.percentile(v, [2.5, 97.5])
        boot.append({"model": m, "metric": k, "Δ": point, "CI 2.5%": lo, "CI 97.5%": hi, "CI chứa 0": lo <= 0 <= hi})
boot = pd.DataFrame(boot)
boot.to_csv(f"{OUT_DIR}/exp006_race_bias-bootstrap.csv", index=False)
print(boot.round(2).to_string(index=False))

# Phụ — Giới tính và tuổi (Ensemble Five, UAR, mean seed)
Cùng code, gần như không tốn thêm công. Nhóm < 100 ảnh chỉ để tham khảo.

In [ ]:
for name, attr_ in [("Giới tính", gender), ("Tuổi", age)]:
    out = {}
    for m, d in HEADS.items():
        out[m] = {g: np.mean([metrics(y[attr_ == g], preds[(m, s, MAIN_PROMPT)][attr_ == g])["UAR"] for s in d])
                  for g in pd.unique(attr_)}
    t = pd.DataFrame(out).T
    t.loc["n"] = pd.Series(attr_).value_counts()
    print(f"--- {name}\n{t.round(2).to_string()}\n")